# Notebook for Lead Time Analysis and Visualization

## Import libs and functions

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np

## Load CSV containing Testset performance per cube and timestep

In [ ]:
REPO = Path.home() / "ARCEME-Vegetation-Recovery"
TABLE_DIR = REPO / "evaluation/analysis_data"
PLOT_DIR = REPO / "evaluation/paper_figures/final_plots/lead_time_analysis"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

data = pd.read_csv(TABLE_DIR / "metrics_detail.csv")
summary = pd.read_csv(TABLE_DIR / "metrics_macro_by_fold.csv")

# Visualization

## General Overview on Cube Basis

In [ ]:
cube_overall = data[(data["scope"]=="overall") & (data["landcover"]=="all")].drop(labels=['configuration', 'split', 'fold', 'scope', 'step', 'landcover', 'policy'], axis=1)
assert cube_overall.shape[0] == 26 * 2
cube_overall.head(4)

In [ ]:
# Check if 5 ccubes excluded
assert (cube_overall[cube_overall["approach"]=="Model"].exclusion_reason.value_counts() == 5).bool

In [ ]:
np.mean(cube_overall[cube_overall["approach"]=="Model"].nnse)

## Lead Time Analysis

In [ ]:
CONFIGURATION = "SGEDConvLSTM RGB+NIR"
SPLIT = "holdout"
FOLD = -1              # Holdout: -1; individual CV fold: 0, 1 or 2

variables = ["nnse", "nse", "mae", "mse", "rmse", "bias", "nnse_raw", "nse_raw", "mae_raw", "mse_raw", "rmse_raw", "bias_raw"]

In [ ]:
lead_time_df = data[
    (data["configuration"] == CONFIGURATION)
    & (data["split"] == SPLIT)
    & (data["fold"] == FOLD)
    & (data["scope"] == "step")
    & (data["landcover"] == "all")
    & (data["approach"].isin(["Model", "Persistence"]))
].copy()
lead_time_df[variables] = lead_time_df[variables].replace([np.inf, -np.inf], np.nan)

In [ ]:
x = lead_time_df[lead_time_df.approach == "Model"]
x.error_exclusion_reason.value_counts()

In [ ]:
# Each metric uses its own available cube scores.
average = (
    lead_time_df.groupby(["approach", "step"])[variables]
    .agg(["mean", "std", "count"])
)
average.columns = [
    f"{variable}_{statistic}"
    for variable, statistic in average.columns
]
average = average.reset_index()

# Split by model and persistence
average_model = average[average["approach"] == "Model"].copy()
average_persistence = average[average["approach"] == "Persistence"].copy()

In [ ]:
# Define if standard deviation should be plotted
std_dev = False

# define if raw is true
raw = True

plot_vars = ["nnse", "nse", "mae", "bias"]

if raw:
    plot_vars = [metric + "_raw" for metric in plot_vars]

labels = {
    "nnse": "NNSE",
    "nse": "NSE",
    "mae": "MAE",
    "mse": "MSE",
    "rmse": "RMSE",
    "bias": "Bias",
    "coverage": "Coverage",
    "n_valid": "Valid vegetation pixels",
}

colors = {"Model": "#0072B2", "Persistence": "#D55E00"}
markers = {"Model": "o", "Persistence": "s"}

# Optional axis limits; None uses automatic scaling.
y_limits = {variable: None for variable in plot_vars}
# y_limits["nnse"] = (0.3, 0.9)
# y_limits["bias"] = (-0.06, 0.02)

steps = np.sort(data["step"].unique())
ncols = 2
nrows = (len(plot_vars) + ncols - 1) // ncols

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(11, 3.5 * nrows),
    squeeze=False,
)

for panel, (ax, variable) in enumerate(zip(axes.flat, plot_vars)):
    counts = {}

    for approach in ["Model", "Persistence"]:
        values = (
            average[average["approach"] == approach]
            .set_index("step")
            .reindex(steps)
        )

        mean = values[f"{variable}_mean"].to_numpy(dtype=float)
        std = values[f"{variable}_std"].to_numpy(dtype=float)
        counts[approach] = (
            values[f"{variable}_count"].fillna(0).astype(int).to_numpy()
        )

        ax.plot(
            steps, mean,
            color=colors[approach],
            marker=markers[approach],
            linestyle="-" if approach == "Model" else "--",
            linewidth=1.8,
            markersize=4.5,
            label=CONFIGURATION if approach == "Model" else "Persistence",
            zorder=3,
        )

        if std_dev:
            ax.fill_between(
                steps, mean - std, mean + std,
                color=colors[approach],
                alpha=0.13,
                linewidth=0,
                zorder=1,
            )

    if variable == "nnse":
        ax.axhline(0.5, color="0.55", linestyle=":", linewidth=1)
    elif variable in ["nse", "bias"]:
        ax.axhline(0, color="0.55", linestyle=":", linewidth=1)

    ax.text(
        1, 1.025,
        "n_cubes: " + " / ".join(counts["Model"][1:].astype(str)),
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=8,
        color="0.35",
    )

    ax.set_xticks(steps)
    ax.set_xlabel("Forecast step", fontsize=10)
    ax.set_ylabel(f"Mean {labels.get(variable, variable)}", fontsize=10)
    ax.set_title(
        f"({chr(97 + panel)}) {labels.get(variable, variable)}",
        loc="left", fontsize=11, fontweight="bold",
    )

    ax.grid(axis="y", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.margins(x=0.05, y=0.12)

    if y_limits[variable] is not None:
        ax.set_ylim(*y_limits[variable])

# Hide unused panels.
for ax in list(axes.flat)[len(plot_vars):]:
    ax.set_visible(False)

handles, legend_labels = axes.flat[0].get_legend_handles_labels()
fig.legend(
    handles, legend_labels,
    loc="upper center", bbox_to_anchor=(0.5, 0.995),
    ncol=2, frameon=False, fontsize=10,
)

note = "Equal cube weights. Counts in right corner refer to number of contributing cubes."
if std_dev:
    note += "\nShading: ±1 SD across contributing cubes."

fig.text(0.5, 0.012, note, ha="center", fontsize=9, color="0.35")
fig.tight_layout(rect=(0, 0.045, 1, 0.95))

configuration_name = CONFIGURATION.replace(" ", "_").replace("+", "_")
suffix = "with_sd" if std_dev else "means_only"
suffix = suffix + "_raw" if raw else suffix
output = PLOT_DIR / f"lead_time_plot_{suffix}.png"

fig.savefig(output, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print(f"Saved: {output}")

## Coverage plot

In [ ]:
from matplotlib.ticker import PercentFormatter, MaxNLocator

coverage_data = lead_time_df[
    (lead_time_df["configuration"] == CONFIGURATION)
    & (lead_time_df["split"] == SPLIT)
    & (lead_time_df["fold"] == FOLD)
    & (lead_time_df["scope"] == "step")
    & (lead_time_df["landcover"] == "all")
    & (lead_time_df["approach"] == "Model")
].copy()

if coverage_data.empty:
    raise ValueError("No rows match the selected dataset.")

if coverage_data.duplicated(["cube_id", "step"]).any():
    raise ValueError("More than one row per cube and step.")

# Use the thresholds recorded in the CSV.
coverage_threshold = coverage_data["min_coverage"].iloc[0]

coverage_data["has_targets"] = coverage_data["n_valid"] > 0
coverage_data["enough_data"] = (
    (coverage_data["n_valid"] >= coverage_data["min_count"])
    & (coverage_data["coverage"] >= coverage_data["min_coverage"])
)

coverage_summary = (
    coverage_data.groupby("step", as_index=False)
    .agg(
        coverage_mean=("coverage", "mean"),
        coverage_median=("coverage", "median"),
        coverage_std=("coverage", "std"),
        n_cubes=("cube_id", "nunique"),
        n_coverage=("coverage", "count"),
        n_with_targets=("has_targets", "sum"),
        n_enough_data=("enough_data", "sum"),
        n_nnse=("nnse", "count"),
    )
    .sort_values("step")
)

coverage_summary

In [ ]:
std_dev = False

steps = coverage_summary["step"].to_numpy()
mean = coverage_summary["coverage_mean"].to_numpy()
median = coverage_summary["coverage_median"].to_numpy()
sd = coverage_summary["coverage_std"].to_numpy()

fig, (ax_cov, ax_count) = plt.subplots(
    2, 1,
    figsize=(9, 7),
    gridspec_kw={"height_ratios": [1.2, 1]},
)

# Coverage across cubes.
ax_cov.plot(
    steps, mean,
    color="#0072B2", marker="o", linewidth=2,
    label="Mean coverage",
)
ax_cov.plot(
    steps, median,
    color="#009E73", marker="D", linestyle="--", linewidth=1.6,
    label="Median coverage",
)

if std_dev:
    ax_cov.fill_between(
        steps, mean - sd, mean + sd,
        color="#0072B2", alpha=0.13, linewidth=0,
        label="±1 SD across cubes",
    )

ax_cov.axhline(
    coverage_threshold,
    color="0.45", linestyle=":", linewidth=1.3,
    label=f"Coverage threshold ({coverage_threshold:.0%})",
)

ax_cov.set_title(
    "(a) Valid observations within the vegetation area",
    loc="left", fontweight="bold", fontsize=11,
)
ax_cov.set_ylabel("Valid vegetation coverage")
ax_cov.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax_cov.legend(frameon=False, fontsize=9, ncol=2)

# Number of cubes available for each evaluation.
width = 0.24

bars = [
    ax_count.bar(
        steps - width, coverage_summary["n_with_targets"],
        width=width, color="#BDBDBD",
        label="At least one valid target",
    ),
    ax_count.bar(
        steps, coverage_summary["n_enough_data"],
        width=width, color="#56B4E9",
        label="Count + coverage criteria met",
    ),
    ax_count.bar(
        steps + width, coverage_summary["n_nnse"],
        width=width, color="#0072B2",
        label="Eligible for NNSE",
    ),
]

for bar_group in bars:
    ax_count.bar_label(bar_group, padding=3, fontsize=9)

ax_count.plot(
    steps, coverage_summary["n_cubes"],
    color="0.4", linestyle=":", linewidth=1.3,
    label="Total cubes",
)

ax_count.set_title(
    "(b) Number of contributing cubes",
    loc="left", fontweight="bold", fontsize=11,
)
ax_count.set_ylabel("Number of cubes")
ax_count.set_xlabel("Forecast step")
ax_count.set_ylim(0, coverage_summary["n_cubes"].max() * 1.25)
ax_count.yaxis.set_major_locator(MaxNLocator(integer=True))
ax_count.legend(frameon=False, fontsize=9, ncol=2)

for ax in (ax_cov, ax_count):
    ax.set_xticks(steps)
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()

PLOT_DIR.mkdir(parents=True, exist_ok=True)
configuration_name = CONFIGURATION.replace(" ", "_").replace("+", "_")
suffix = "with_sd" if std_dev else "means_only"

output = PLOT_DIR / (
    f"coverage_{suffix}.png"
)

fig.savefig(output, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print(f"Saved: {output}")